# 09 — Fatoração de matrizes e SVD

**EP · Data Science Knowledge Base | Sistemas de Recomendação | Notebook 09/16**

**Execução:** Reinicie o kernel e execute as células em ordem. Observe as instruções específicas e as dependências opcionais indicadas neste notebook.


## Materiais complementares

- [Guia Técnico](reference_materials/09_svd_fatoracao.pdf)
- [Cheat Sheet](cheat_sheets/cheat_sheet_09_svd_fatoracao.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
- Compreender fatores latentes
- Aplicar TruncatedSVD como demonstração, não como Surprise SVD

## Fundamentação conceitual
Na fatoração, aproximamos $R\approx PQ^T$. `TruncatedSVD` de uma matriz com ausências preenchidas por zero é **demonstração algébrica**, não equivalente ao SVD enviesado do pacote Surprise, que minimiza erro apenas nas avaliações observadas.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
DATA = Path('datasets')
if not (DATA / 'ratings.csv').exists():
    raise FileNotFoundError('Abra o JupyterLab na pasta da Disciplina 09 (onde existe datasets/).')
ratings = pd.read_csv(DATA / 'ratings.csv')
movies = pd.read_csv(DATA / 'movies.csv')
print('Avaliações:', len(ratings), '| Filmes:', len(movies), '| Usuários:', ratings.userId.nunique())

Avaliações: 100836 | Filmes: 9742 | Usuários: 610


## Fatores latentes em matriz pequena

In [2]:
from sklearn.decomposition import TruncatedSVD
popular=ratings.movieId.value_counts().head(250).index
M=ratings[ratings.movieId.isin(popular)].pivot_table(index='userId',columns='movieId',values='rating').fillna(0)
model=TruncatedSVD(n_components=12,random_state=42)
U=model.fit_transform(M); reconstruction=U @ model.components_
print('Original:',M.shape,'Fatores:',U.shape,'Reconstrução:',reconstruction.shape)

Original: (602, 250) Fatores: (602, 12) Reconstrução: (602, 250)


**Interpretação.** A matriz reconstruída não é automaticamente uma previsão calibrada de notas.

## Recomendações latentes demonstrativas

In [3]:
user=1; i=M.index.get_loc(user); scores=pd.Series(reconstruction[i],index=M.columns)
seen=set(ratings.loc[ratings.userId.eq(user),'movieId']); top=scores.drop(labels=list(seen.intersection(scores.index))).nlargest(5)
print(movies[movies.movieId.isin(top.index)][['movieId','title']].to_string(index=False))

 movieId                                     title
      32 Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
     589         Terminator 2: Judgment Day (1991)
    1036                           Die Hard (1988)
    1200                             Aliens (1986)
    2762                   Sixth Sense, The (1999)


**Interpretação.** Pontuações latentes geram candidatos, mas requerem teste offline honesto.

## Aprofundamento e aplicação

**Ponto de atenção deste capítulo:** fatores e regularização. Este complemento não substitui as seções e exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **fatores e regularização** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **fatores e regularização** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.


## Análise crítica — SVD algébrica versus fatoração colaborativa

`TruncatedSVD` aplicada à matriz preenchida com zero minimiza uma aproximação matricial, **não** o erro apenas nas notas observadas. A fatoração colaborativa convencional ajusta fatores `p_u`, `q_i` e eventualmente vieses minimizando perda nos pares de treino observados. Portanto, não se deve tratar a reconstrução atual como uma previsão de notas validada.

**Origem:** aprofundamento técnico autoral fundamentado no experimento deste notebook. Não é transcrição dos materiais históricos.

In [4]:
# Medir a reconstrução SOMENTE nos pares que já eram observados — diagnóstico de treino
observado = M.to_numpy() > 0
r = M.to_numpy()
rmse_observado_treino = np.sqrt(np.mean((reconstruction[observado]-r[observado])**2))
print('RMSE nos pares observados usados no próprio ajuste (NÃO teste):', round(float(rmse_observado_treino),3))
print('Esse valor não comprova generalização nem equivalência ao Surprise SVD.')

RMSE nos pares observados usados no próprio ajuste (NÃO teste): 1.937
Esse valor não comprova generalização nem equivalência ao Surprise SVD.


### Critério de leitura

Interprete os resultados como diagnóstico didático. Para comparar modelos de recomendação, defina previamente população, itens elegíveis, janela temporal, sinal de relevância, baseline e métrica; mantenha treino, validação e teste separados.

**Atividade:** registre o efeito de uma mudança de parâmetro e explique uma ameaça à validade do experimento.

<!-- EP_FASE2_09_LOTE1 -->
### Experimento complementar — fatoração ajustada somente às notas observadas

O `TruncatedSVD` acima é útil para estudar a decomposição da matriz preenchida com zeros, **mas não otimiza apenas as avaliações observadas**. A comparação abaixo ajusta fatores latentes com gradiente estocástico apenas sobre avaliações no **treino**. O conjunto de teste é reservado antes do ajuste; comparamos RMSE com uma previsão constante pela média **do treino**, nos mesmos exemplos de teste com usuário e filme conhecidos.

**Escopo e limitações:** estudo didático com apenas os 250 filmes mais frequentes **no treino**, escolhidos após a divisão inicial para não consultar o teste na seleção do catálogo. Não é um benchmark geral: a população de itens avaliados é restrita aos filmes selecionados no treino. A divisão é aleatória por interação, não temporal; o teste só considera pares conhecidos no treino, e a cobertura é informada. Sementes e hiperparâmetros são fixos para reprodutibilidade pedagógica, não foram selecionados por validação. A previsão é limitada à faixa de notas de 0,5 a 5,0. Nenhum resultado aqui representa desempenho de produção ou equivalência ao algoritmo `SVD` do Surprise.


In [5]:
from sklearn.model_selection import train_test_split
# Divisão antes da seleção de filmes: o teste não informa a seleção do catálogo.
ep_avaliacoes = ratings[['userId', 'movieId', 'rating']].drop_duplicates(['userId', 'movieId'])
ep_treino_total, ep_teste_total = train_test_split(
    ep_avaliacoes, test_size=0.20, random_state=42
)
ep_populares_treino = ep_treino_total.movieId.value_counts().head(250).index
ep_treino = ep_treino_total.loc[ep_treino_total.movieId.isin(ep_populares_treino)]
ep_teste = ep_teste_total.loc[ep_teste_total.movieId.isin(ep_populares_treino)]
ep_users = pd.Index(sorted(ep_treino.userId.unique()))
ep_items = pd.Index(sorted(ep_treino.movieId.unique()))
ep_u_treino = ep_users.get_indexer(ep_treino.userId)
ep_i_treino = ep_items.get_indexer(ep_treino.movieId)
ep_notas_treino = ep_treino.rating.to_numpy(dtype=float)
ep_mu = float(ep_notas_treino.mean())
ep_rng = np.random.default_rng(42)
ep_k = 8
ep_P = ep_rng.normal(0, 0.08, (len(ep_users), ep_k))
ep_Q = ep_rng.normal(0, 0.08, (len(ep_items), ep_k))
ep_bu = np.zeros(len(ep_users))
ep_bi = np.zeros(len(ep_items))
ep_eta, ep_reg, ep_epochs = 0.01, 0.05, 10
for ep_epoch in range(ep_epochs):
    for ep_j in ep_rng.permutation(len(ep_notas_treino)):
        ep_u, ep_i = ep_u_treino[ep_j], ep_i_treino[ep_j]
        ep_p = ep_P[ep_u].copy()
        ep_q = ep_Q[ep_i].copy()
        ep_prev = ep_mu + ep_bu[ep_u] + ep_bi[ep_i] + ep_p @ ep_q
        ep_erro = ep_notas_treino[ep_j] - ep_prev
        ep_bu[ep_u] += ep_eta * (ep_erro - ep_reg * ep_bu[ep_u])
        ep_bi[ep_i] += ep_eta * (ep_erro - ep_reg * ep_bi[ep_i])
        ep_P[ep_u] += ep_eta * (ep_erro * ep_q - ep_reg * ep_p)
        ep_Q[ep_i] += ep_eta * (ep_erro * ep_p - ep_reg * ep_q)
# Avaliamos somente pares para os quais ambos os IDs foram vistos no treino.
ep_u_teste = ep_users.get_indexer(ep_teste.userId)
ep_i_teste = ep_items.get_indexer(ep_teste.movieId)
ep_warm = (ep_u_teste >= 0) & (ep_i_teste >= 0)
assert ep_warm.any(), 'Nenhum par conhecido ficou disponível no teste.'
ep_uu, ep_ii = ep_u_teste[ep_warm], ep_i_teste[ep_warm]
ep_y_teste = ep_teste.loc[ep_warm, 'rating'].to_numpy(dtype=float)
ep_previsoes = np.clip(
    ep_mu + ep_bu[ep_uu] + ep_bi[ep_ii] + np.sum(ep_P[ep_uu] * ep_Q[ep_ii], axis=1),
    0.5, 5.0
)
ep_rmse_mf = float(np.sqrt(np.mean((ep_y_teste - ep_previsoes)**2)))
ep_rmse_media = float(np.sqrt(np.mean((ep_y_teste - ep_mu)**2)))
print(f'Treino: {len(ep_treino)} avaliações; teste: {len(ep_teste)} avaliações')
print(f'Cobertura de pares conhecidos no teste: {ep_warm.mean():.1%}')
print(f'RMSE teste (média do treino): {ep_rmse_media:.4f}')
print(f'RMSE teste (fatores treinados só em notas observadas): {ep_rmse_mf:.4f}')
print('Nenhum desses resultados usa o RMSE de reconstrução em treino acima como teste.')


Treino: 23715 avaliações; teste: 5802 avaliações
Cobertura de pares conhecidos no teste: 100.0%
RMSE teste (média do treino): 0.9640
RMSE teste (fatores treinados só em notas observadas): 0.8246
Nenhum desses resultados usa o RMSE de reconstrução em treino acima como teste.


## Limitações e boas práticas
Não interprete zeros imputados como avaliações reais nem reporte RMSE desse exemplo como qualidade geral.

## Exercícios
Compare os resultados para 6 e 20 componentes.

## Soluções comentadas
Altere `n_components` respeitando `min(M.shape)`; mais fatores podem aumentar sobreajuste.

## Síntese
Reveja os resultados e compare métodos usando um protocolo de dados e métricas consistente. Não trate avaliações ausentes como avaliações negativas.

### Exportação de resultados

A próxima célula grava os resultados do capítulo em `outputs/`.


In [6]:
# Exportação de resultados do capítulo
# Execute todas as células em sequência antes de exportar.
# Cada arquivo documenta um experimento ou protocolo distinto.
from pathlib import Path
out = Path('outputs')
out.mkdir(parents=True, exist_ok=True)
top.rename("score_latente_demonstrativo").rename_axis("movieId").reset_index().merge(
    movies[["movieId", "title"]], on="movieId", how="left").to_csv(
    out / "09_top_latente_demonstrativo.csv", index=False)

# Resultados das análises metodológicas complementares
_ep_df = pd.DataFrame([{
    'metodo': 'sgd_fatores_latentes_so_notas_observadas',
    'n_treino_filmes_populares': int(len(ep_treino)),
    'n_teste_filmes_populares': int(len(ep_teste)),
    'n_teste_warm': int(ep_warm.sum()),
    'cobertura_warm_no_teste_filtrado': float(ep_warm.mean()),
    'rmse_teste_warm_fatores': float(ep_rmse_mf),
    'rmse_teste_warm_media_treino': float(ep_rmse_media),
    'fatores_latentes': int(ep_k), 'epocas': int(ep_epochs),
    'seed': 42,
    'escopo': 'teste_aleatorio_apenas_pares_warm_sem_hiperparametrizacao'
}])
_ep_destino = out / '09_rmse_fatores_observados_holdout.csv'
_ep_df.to_csv(_ep_destino, index=False)
print('Arquivo exportado:', _ep_destino)


Arquivo exportado: outputs/09_rmse_fatores_observados_holdout.csv
